# X (Twitter) Profile Image Scraper
This notebook contains functions to extract the high-resolution (`400x400`) profile picture URLs for one or multiple X profiles.

In [ ]:
import re
import requests
from bs4 import BeautifulSoup
from typing import List, Dict
import pandas as pd

def get_x_profile_image_url(profile_url: str) -> str:
    """
    Given a single X (Twitter) profile URL, scrape the page and return the direct
    400x400 profile image URL.
    """
    profile_url = profile_url.strip()
    if "twitter.com" in profile_url:
        profile_url = profile_url.replace("twitter.com", "x.com")
        
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    response = requests.get(profile_url, headers=headers)
    if response.status_code != 200:
        raise ValueError(f"Failed to fetch profile page. Status code: {response.status_code}")
        
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find the OpenGraph image meta tag which holds the profile image url
    og_image_tag = soup.find('meta', property='og:image')
    if not og_image_tag or not og_image_tag.get('content'):
        raise ValueError("Profile image URL not found in metadata. The account might be private or suspended.")
        
    avatar_url = og_image_tag['content']
    
    # Convert size suffix to _400x400 to get high resolution
    suffix_pattern = r'_(?:normal|bigger|mini|reasonably_small|x96|\d+x\d+)(\.[a-zA-Z0-9]+)$'
    if re.search(suffix_pattern, avatar_url):
        avatar_url = re.sub(suffix_pattern, r'_400x400\1', avatar_url)
        
    return avatar_url

def get_multiple_x_profile_image_urls(profile_urls: List[str]) -> Dict[str, str]:
    """
    Given a list of X profile URLs, scrape them and return a dictionary mapping
    each profile URL to its 400x400 profile image URL (or error message).
    """
    results = {}
    for url in profile_urls:
        url = url.strip()
        if not url:
            continue
        try:
            img_url = get_x_profile_image_url(url)
            results[url] = img_url
        except Exception as e:
            results[url] = f"Error: {str(e)}"
    return results

### Option 1: Process a Single URL

In [ ]:
url = "https://x.com/win_azurite"
try:
    img_url = get_x_profile_image_url(url)
    print(f"Success!\nProfile: {url}\nImage:   {img_url}")
except Exception as e:
    print(f"Error: {e}")

### Option 2: Process Multiple URLs & Display in Excel-Friendly Format
You can input your list of URLs below (either as a Python list or newline-separated text block).

In [ ]:
# Input multiple URLs here (one URL per line)
urls_input = """
https://x.com/win_azurite
https://x.com/elonmusk
https://x.com/jack
https://x.com/nasa
"""

# Parse the input string into a list of URLs
urls_list = [line.strip() for line in urls_input.strip().split('\n') if line.strip()]

print(f"Processing {len(urls_list)} URLs...")
batch_results = get_multiple_x_profile_image_urls(urls_list)

# 1. Create a pandas DataFrame (Jupyter renders this as a clean HTML table for direct copy-paste to Excel)
df = pd.DataFrame(list(batch_results.items()), columns=['X Profile URL', 'Profile Image URL'])

# Display standard Jupyter HTML table
df

In [ ]:
# 2. Fallback: Print Tab-Separated Values (TSV) which can be copied and pasted directly into Excel
print("--- TSV Format (Select and Copy the rows below, then Paste into Excel) ---")
print(df.to_csv(sep='\t', index=False))